[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/intronlp-2026/blob/main/notebooks/week-06.ipynb)

# 6주차 실습: 단어를 벡터로 - 동시발생 행렬과 코사인 유사도

**목표.** 작은 말뭉치에서 단어가 함께 다니는 단어를 세어 행렬을 만들고, 행을 단어 벡터로 써서 두 단어가 얼마나 비슷한지 숫자로 재 본다.

## 0. 준비

표를 보여 줄 pandas를 설치합니다. Colab에는 이미 설치되어 있어 바로 넘어갑니다.

설치 셀에서 오류가 나면 다음 셀로 넘어가지 말고 오류의 마지막 줄을 확인하세요.

In [ ]:
%pip -q install "pandas>=1.5.3"

In [ ]:
import sys
import math
import pandas

print("Python:", sys.version.split()[0])
print("pandas:", pandas.__version__)
print("설치 확인 끝")

## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.

이 노트북의 문장은 **설명용 예시**입니다. 제주 이야기 문장 13개로 작은 말뭉치를 직접 만듭니다. 실제 방언 자료(AI Hub 한국어 방언 발화, 제주도)는 각자 승인을 받아야 하고 저장소에 올릴 수 없어서, 1-6에서는 방언 표본도 직접 만든 문장으로 봅니다.

### 1-1. 제주 이야기 문장 13개

바다, 하늘, 귤, 음식에 대한 문장 13개입니다. 단어가 겹치는 자리를 눈여겨보세요. `푸르다` 는 바다와 하늘 문장에, `먹었다` 는 음식 문장에 나옵니다.

In [ ]:
sentences = [
    "제주 바다가 아주 푸르다",
    "가을 하늘이 아주 푸르다",
    "우도 바다가 더 맑다",
    "성산 바다가 잔잔하다",
    "겨울 하늘에 별이 잔뜩 있다",
    "새별오름에서 노을이 붉다",
    "한라산 오름에 오르면 뷰가 좋다",
    "귤밭에서 귤을 따왔다",
    "겨울 귤은 달고 과즙이 많다",
    "감귤 상자를 포장해 선물했다",
    "고기국수를 뜨겁게 먹었다",
    "갈치조림을 뜨끈하게 먹었다",
    "귤을 깨끗이 씻어 먹었다"    ]

print("문장", len(sentences), "개")
for i, s in enumerate(sentences):
    print(f"  {i+1:2d}. {s}")

vocab = sorted({w for s in sentences for w in s.split()})
print("\n전체 단어:", len(vocab), "개")

### 1-2. 창(window)으로 문맥 보기

분포 가설은 이렇게 말합니다. **단어의 뜻은 그 단어가 함께 다니는 단어에 있다.** 함께 다닌다는 것을 재는 방법이 창(window)입니다. 기준 단어에서 왼쪽 오른쪽으로 몇 칸을 볼지 정합니다.

아래 셀은 기준 단어 주위를 창으로 훑어 이웃 단어를 보여 줍니다. 창 1은 바로 옆 한 칸, 창 2는 옆 두 칸입니다. 창을 넓히면 이웃이 늘어나는지 확인하세요.

In [ ]:
def show_window(sentence, word, window):
    ws = sentence.split()
    i = ws.index(word)
    lo = max(0, i - window)
    hi = min(len(ws), i + window + 1)
    marks = []
    for j in range(len(ws)):
        if j == i:
            marks.append("[" + ws[j] + "]")
        elif lo <= j < hi:
            marks.append("(" + ws[j] + ")")
        else:
            marks.append(ws[j])
    neighbors = [ws[j] for j in range(len(ws)) if lo <= j < hi and j != i]
    print(f"창 {window}: {' '.join(marks)}")
    print(f"  이웃: {', '.join(neighbors) if neighbors else '없음'}")


show_window("고기국수를 뜨겁게 먹었다", "먹었다", 1)
show_window("고기국수를 뜨겁게 먹었다", "먹었다", 2)
print()
show_window("귤밭에서 귤을 따왔다", "귤을", 1)
show_window("귤밭에서 귤을 따왔다", "귤을", 2)

### 1-3. 창을 세면 행렬이 된다

이웃을 눈으로만 보지 않고 **세어서 표**로 만듭니다. 두 단어가 한 문장에서 창 안에 함께 들어온 횟수를 칸에 씁니다. 이 표가 **동시발생 행렬**입니다. 3주차에 단어 짝을 세었듯이, 이번 주는 창 안의 단어 쌍을 셉니다.

행렬의 한 행이 단어 하나의 기록입니다. `귤을` 의 행을 봅니다. 귤을과 창 안에서 함께 나온 단어 넷이 기록되어 있습니다. 나머지 38칸은 모두 0입니다. 단어 42개 중 넷과만 함께 다녔기 때문입니다.

마지막 표는 42 x 42 행렬에서 단어 여덟 개만 잘라 본 것입니다. 표는 대각선을 기준으로 좌우가 같습니다. `바다가` 와 `푸르다` 가 함께 나왔다면 `푸르다` 와 `바다가` 도 함께 나온 것이기 때문입니다.

In [ ]:
def build_matrix(sentences, window=2):
    vocab = sorted({w for s in sentences for w in s.split()})
    idx = {w: i for i, w in enumerate(vocab)}
    M = [[0] * len(vocab) for _ in vocab]
    for s in sentences:
        ws = s.split()
        for i, w in enumerate(ws):
            for d in range(1, window + 1):
                if i + d < len(ws):
                    a, b = idx[w], idx[ws[i + d]]
                    M[a][b] += 1
                    M[b][a] += 1
    return vocab, M


def row_vector(vocab, M, word):
    i = vocab.index(word)
    return {vocab[j]: M[i][j] for j in range(len(vocab)) if M[i][j] > 0}


vocab, matrix = build_matrix(sentences, window=2)
n = len(vocab)
nonzero = sum(1 for row in matrix for v in row if v > 0)
print(f"동시발생 행렬: {n} x {n}, 칸 {n*n}개")
print(f"0이 아닌 칸: {nonzero}개 ({nonzero/(n*n)*100:.1f}%)")
print()
print("귤을의 행:", row_vector(vocab, matrix, "귤을"))
print("바다가의 행:", row_vector(vocab, matrix, "바다가"))

# 42 x 42 표 전체는 너무 넓어서, 단어 여덟 개만 골라 표로 봅니다
import pandas as pd
show = ["바다가", "하늘이", "아주", "푸르다", "귤을", "귤은", "먹었다", "고기국수를"]
pd.DataFrame(matrix, index=vocab, columns=vocab).loc[show, show]

### 1-4. 행의 한 줄이 단어 벡터다

행렬의 한 행을 **단어 벡터**로 씁니다. 좌표가 단어 개수만큼 있는 숫자 목록이고, 대부분이 0인 희소한 벡터입니다. 이제 단어를 눈으로 비교하는 대신 **숫자 목록을 비교**할 수 있습니다.

두 단어의 벡터에서 **겹치는 좌표**를 봅니다. 귤을과 귤은은 사람에게는 같은 과일이지만 겹치는 좌표가 하나도 없습니다. 바다가와 하늘이는 겹치는 좌표가 있습니다. 이 차이가 다음 단계의 유사도에 그대로 드러납니다.

In [ ]:
pairs = ["귤을", "귤은", "감귤", "바다가", "하늘이"]
for w in pairs:
    print(f"{w}: {row_vector(vocab, matrix, w)}")

def overlap(a, b):
    va, vb = row_vector(vocab, matrix, a), row_vector(vocab, matrix, b)
    return sorted(set(va) & set(vb))

print()
print("귤을과 귤은이 함께 다니는 단어:", overlap("귤을", "귤은") or "없음")
print("귤을과 감귤이 함께 다니는 단어:", overlap("귤을", "감귤") or "없음")
print("바다가와 하늘이가 함께 다니는 단어:", overlap("바다가", "하늘이"))

### 1-5. 코사인 유사도로 재기

벡터 두 개가 얼마나 같은 방향을 보는지 재는 값이 **코사인 유사도**입니다. 같은 방향이면 1, 직각이면 0, 반대 방향이면 -1입니다. 두 단어의 벡터가 겹치는 좌표를 많이 가질수록 값이 1에 가까워집니다.

먼저 좌표 두 개짜리 벡터로 감을 잡고, 단어 벡터에 적용합니다. 마지막 표가 이 노트북의 결론입니다. **사람에게 비슷한 단어가 숫자로도 비슷하게 나오는 경우와, 그렇지 않은 경우가 같은 말뭉치 안에 함께 있습니다.**

맨 마지막 목록은 말뭉치 전체에서 유사도가 가장 높은 쌍입니다. `맑다` 와 `우도` 가 1.00으로 나오는데, 뜻이 같아서가 아닙니다. "우도 바다가 더 맑다" 한 문장에서 두 단어의 이웃이 똑같이 `바다가` 와 `더` 이기 때문입니다. **문장이 적으면 한 문장이 유사도를 통째로 정합니다.**

In [ ]:
def cosine(a, b):
    dot = sum(a.get(w, 0) * b.get(w, 0) for w in set(a) | set(b))
    na = math.sqrt(sum(x * x for x in a.values()))
    nb = math.sqrt(sum(x * x for x in b.values()))
    return 0.0 if na == 0 or nb == 0 else dot / (na * nb)


print("좌표 2개짜리 예시 벡터")
print(f"[2, 1] ~ [1, 2]: {cosine({0: 2, 1: 1}, {0: 1, 1: 2}):.2f} (같은 방향쯤)")
print(f"[2, 1] ~ [-1, 2]: {cosine({0: 2, 1: 1}, {0: -1, 1: 2}):.2f} (직각)")
print(f"[2, 1] ~ [-2, -1]: {cosine({0: 2, 1: 1}, {0: -2, 1: -1}):.2f} (반대)")

print()
print("단어 벡터의 유사도")
word_pairs = [
    ("바다가", "하늘이"), ("고기국수를", "갈치조림을"),
    ("먹었다", "귤을"), ("먹었다", "고기국수를"),
    ("귤을", "귤은"), ("귤을", "감귤"),
    ("귤은", "감귤"), ("귤은", "하늘에"),
]
for a, b in word_pairs:
    va, vb = row_vector(vocab, matrix, a), row_vector(vocab, matrix, b)
    print(f"  {a} ~ {b}: {cosine(va, vb):.2f}")

print()
sims = sorted(((cosine(row_vector(vocab, matrix, "귤을"), row_vector(vocab, matrix, o)), o)
               for o in vocab if o != "귤을"), reverse=True)
print("귤을과 가장 비슷한 단어 세 개:")
for s, o in sims[:3]:
    print(f"  {o}: {s:.2f}")

print()
from itertools import combinations
best = sorted(((cosine(row_vector(vocab, matrix, a), row_vector(vocab, matrix, b)), a, b)
               for a, b in combinations(vocab, 2)), reverse=True)
print("말뭉치 전체에서 가장 비슷한 쌍 세 개:")
for s, a, b in best[:3]:
    print(f"  {a} ~ {b}: {s:.2f}")

### 1-6. 같은 뜻, 다른 표기: 방언과 표준어

제주어 `바당` 은 표준어 `바다` 입니다. 사람에게는 같은 뜻이지만, 동시발생 행렬은 두 단어를 다른 칸으로 셉니다. 두 단어가 비슷하게 나오려면 **이웃이 겹쳐야** 합니다.

아래는 직접 만든 설명용 표본입니다. 방언 문장과 표준어 문장을 한 쌍씩 두고, 주변 말이 얼마나 방언인지에 따라 `바당에서` 와 `바다에서` 의 유사도가 어떻게 달라지는지 봅니다. 어휘는 `하영` (많이), `놀멍` (놀며), `쉬멍` (쉬며)입니다.

In [ ]:
dialect_cases = {
    "이웃이 같다": ["바당에서 물질을 했다", "바다에서 물질을 했다"],
    "이웃 하나가 방언": ["바당에서 하영 놀았다", "바다에서 많이 놀았다"],
    "이웃도 모두 방언": ["바당에서 하영 놀멍 쉬멍", "바다에서 많이 놀며 쉬며"],
}

for name, pair in dialect_cases.items():
    d_vocab, d_matrix = build_matrix(pair, window=2)
    a = row_vector(d_vocab, d_matrix, "바당에서")
    b = row_vector(d_vocab, d_matrix, "바다에서")
    print(f"{name}: 바당에서 ~ 바다에서 = {cosine(a, b):.2f}")
    print(f"  바당에서의 이웃 {sorted(a)} / 바다에서의 이웃 {sorted(b)}")

all_sentences = [s for pair in dialect_cases.values() for s in pair]
d_vocab, d_matrix = build_matrix(all_sentences, window=2)
mixed = cosine(row_vector(d_vocab, d_matrix, "바당에서"), row_vector(d_vocab, d_matrix, "바다에서"))
print()
print(f"여섯 문장을 한 말뭉치에 섞으면: 바당에서 ~ 바다에서 = {mixed:.3f}")

## 2. 한 지점만 바꿔 보기

아래 셀의 `TODO` 로 표시된 **한 곳만** 바꾸고 다시 실행하세요.

> 바꾸기 전 결과를 먼저 확인해 두면 무엇이 달라졌는지 비교할 수 있습니다.

창 크기를 1에서 3으로 바꿔 봅니다. 창이 좁아지면 이웃이 줄고, 넓어지면 늘어납니다. 행렬의 0이 아닌 칸 수와 단어 유사도가 어떻게 달라지는지 눈여겨보세요.

특히 `고기국수를` 과 `갈치조림을` 의 유사도를 지켜보세요. 창 1에서는 두 단어가 이웃을 공유하지 못하지만, 창 2부터는 `먹었다` 를 함께 다니는 단어로 세어 유사도가 올라갑니다.

In [ ]:
# TODO: 창 크기를 바꿔 보세요 (1, 2, 3)
window = 2

# 아래는 그대로 둡니다
vocab, matrix = build_matrix(sentences, window)
n = len(vocab)
nonzero = sum(1 for row in matrix for v in row if v > 0)
print(f"창 {window}: 행렬 {n} x {n}, 0이 아닌 칸 {nonzero}개 ({nonzero/(n*n)*100:.1f}%)")
print()
check_pairs = [
    ("고기국수를", "갈치조림을"), ("바다가", "하늘이"),
    ("귤을", "귤은"), ("귤을", "감귤"),
]
for a, b in check_pairs:
    va, vb = row_vector(vocab, matrix, a), row_vector(vocab, matrix, b)
    print(f"  {a} ~ {b}: {cosine(va, vb):.2f}")
print()
print("귤을의 이웃:", row_vector(vocab, matrix, "귤을"))

## 3. 확인 질문

1. 1-3 출력에서 `귤을` 의 이웃 단어 넷을 옮기고, `귤은` 의 이웃과 비교해 겹치는 단어가 없음을 확인하세요.
2. TODO 셀에서 창을 1로 바꾸면 `고기국수를` 과 `갈치조림을` 의 유사도가 어떻게 되나요? 이유를 창과 이웃 단어로 설명하세요.
3. 1-6에서 `바당에서` ~ `바다에서` 가 1.00이 되는 경우와 0.00이 되는 경우를 옮기고, 무엇이 둘을 갈랐는지 적으세요.
4. `귤을`, `귤은`, `감귤` 은 사람에게 같은 과일인데 유사도가 모두 0.00입니다. 이런 어긋남을 줄이려면 무엇이 필요할까요? 힌트는 4주차입니다.

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.

*(여기에 답을 적으세요)*

## 4. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/intronlp-2026)의 `assignments/week-06/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.